In [ ]:
import os

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")  # Set key via environment variable
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

In [ ]:
# from langchain.llms import OpenAI # this code has been deprecated since recording.
# ...existing code...
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, NonNegativeInt
from typing import List
from random import sample
# ...existing code...

First, let's create a loader and load reviews from tv-reviews.csv into memory

In [ ]:
# TODO: load reviews from tv-reviews.csv

from pathlib import Path
import pandas as pd
from langchain_core.documents import Document

csv_path = Path("tv-reviews.csv")
df = pd.read_csv(csv_path)

# Use the first text-like column if present, else stringify the row
text_col = next((c for c in ["review", "text", "content"] if c in df.columns), None)
if text_col:
    reviews = [Document(page_content=str(v)) for v in df[text_col].fillna("")]
else:
    reviews = [Document(page_content=str(row.to_dict())) for _, row in df.iterrows()]

print(f"Loaded {len(reviews)} reviews")

Then, let's initialize our LLM

In [ ]:
# TODO: initialize OpenAI object with your API key

Now, let's setup our parser and a template  - 

**Note**  that since recording, the code to initialize the model has been updated to 

`llm = ChatOpenAI()`

In [ ]:
class ReviewSentiment(BaseModel):
    positives: List[NonNegativeInt] = Field(description="index of a positive TV review, starting from 0")
    negatives: List[NonNegativeInt] = Field(description="index of a negative TV review, starting from 0")
        
parser = PydanticOutputParser(pydantic_object=ReviewSentiment)

# Setup a template with partial and input variables
template = """
{format_instructions}

Context:
{context}

Question: {question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

Pick 3 sample reviews to classify - LLMs have a limited context window they can work with. In later exercises, we'll see how to deal with that differently

In [ ]:
import os

# Pick 3 random reviews and save them into reviews_to_classify variable
reviews_to_classify = sample(reviews, min(3, len(reviews)))
context = "\n\n".join([f"Review {i}: {doc.page_content}" for i, doc in enumerate(reviews_to_classify)])

question = """
    Review TVs provided in the context. 
    Only use the reviews provided in this context, do not make up new reviews or use any existing information you know about these TVs. 
    If there are no positive or negative reviews, output an empty JSON array. 
"""
query = prompt.format(context=context, question=question)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0,
    max_tokens=500,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE"),
)

# generate textual prompt from the prompt template
question = """
    Review TVs provided in the context. 
    Only use the reviews provided in this context, do not make up new reviews or use any existing information you know about these TVs. 
    If there are no positive or negative reviews, output an empty JSON array. 
"""
query = prompt.format(context = context, question = question)

Finally, let's send our query to LLM and use the parser we setup to parse an output into a Python object

**NOTE**: Since recording the code to feed the query to the llm has been updated to

`llm.predict(query)`

In [ ]:
# TODO: query LLM, then parse output into the result variable
response = llm.invoke(query)
result = parser.parse(response.content)

print("Positives:\n" + "\n".join([reviews_to_classify[i].page_content for i in result.positives]))
print("Negatives:\n" + "\n".join([reviews_to_classify[i].page_content for i in result.negatives]))